# Mechanism analysis: RQ3

*Which runtime characteristics account for the energy differences between self-defense techniques?*

Feeds: **§6.4 Mechanism (RQ3)** (Figure 3). Inputs are the per-cell logcat activity tables (attribution by process ID, not string
match: `tools/logcat_app_parser.py`) and the static code measurements in `6_results_data/inputs/allatori_static_code_effects.csv`,
which no notebook regenerates (see its PROVENANCE file).

The logcat tables hold every cell with a capture, valid or not. They are **restricted here to the same cells the energy
analysis uses**: the filter chain and the **three series batches** (the pilot enters no Results number), so a crashing build cannot contribute runtime behaviour that the energy figures
exclude. Without this, documenter's non-working Allatori build shows ~+39 threads of crash-and-relaunch churn.

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import thesis_lib as T

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
REF = T.reference()
for d in (T.TABLES, T.FIGURES, T.SUPERVISOR):
    d.mkdir(exist_ok=True)
T.figure_style()
NB = "notebooks/Mechanism_Analysis.ipynb"

In [2]:
frames = []
for b in T.SERIES:                                    # the pilot enters no Results number
    d = pd.read_csv(T.DATA / "inputs" / f"logcat_activity_{b}.csv")
    d["batch"] = b; frames.append(d)
lc = pd.concat(frames, ignore_index=True)
lc["rep"] = lc["rep"].astype(str)
print(f"{len(lc)} captured cells before restriction")
df, _ = T.apply_filters(T.load_raw())
series = df[df["batch"].isin(T.SERIES)].copy()
key = ["batch", "phone", "app", "variant", "rep"]
lc = lc.merge(series[key].drop_duplicates(), on=key, how="inner")
print(f"{len(lc)} captured cells that are also in the analysed set, across {lc['app'].nunique()} applications")

3597 captured cells before restriction
2645 captured cells that are also in the analysed set, across 9 applications


In [3]:
# How many cells have a system-log capture of the measured run (logcat_activity.PROVENANCE.md). In series 1, capture
# ran throughout the batch, the 13 August session included.
raw = T.load_raw()
cov = pd.DataFrame({"cells_run": raw[raw["batch"].isin(T.SERIES)].groupby("batch").size(),
                    "cells_with_log": pd.concat(frames).groupby("batch").size(),
                    "analysed_cells": series.groupby("batch").size(),
                    "analysed_with_log": lc.groupby("batch").size()}).astype(int)
cov.loc["all series"] = cov.sum()
cov.index.name = "batch"
cov.to_csv(T.TABLES / "log_coverage.csv")
T.register("tables/log_coverage.csv", "REPO", "5 Execution / Methods (log coverage)",
           "cells with a system-log capture of the measured run, per series batch, of all cells and of the analysed cells", NB)
print(cov.to_string())
T.record("log_coverage", series1_cells_with_log=f"{cov.loc['series1', 'cells_with_log']} of {cov.loc['series1', 'cells_run']}",
         analysed_cells_with_log=f"{cov.loc['all series', 'analysed_with_log']} of {cov.loc['all series', 'analysed_cells']}")

            cells_run  cells_with_log  analysed_cells  analysed_with_log
batch                                                                   
series1          1200            1200             884                884
series2          1200            1199             879                878
series3          1200            1198             884                883
all series       3600            3597            2647               2645


## §6.4 (RQ3): The freeRASP runtime fingerprint

Cells whose application-attributed logcat contains at least one `auditd` (SELinux denial) line. The application's
processes are its main process and its own sub-processes (`logcat_activity.PROVENANCE.md`); the
count from the main process alone is reported beside it. Outside freeRASP, every denial comes from one process,
documenter's conversion worker, which logs one in every build (a read of the root directory); the cell below checks it.


In [4]:
lc["has_auditd"] = lc["auditd_lines"].fillna(0) > 0
lc["has_auditd_main"] = lc["auditd_lines_main"].fillna(0) > 0
aud = lc.groupby("variant").agg(cells=("has_auditd", "size"), with_auditd=("has_auditd", "sum"),
                                with_auditd_main_process=("has_auditd_main", "sum"))
aud["share_pct"] = 100 * aud["with_auditd"] / aud["cells"]
aud["share_main_process_pct"] = 100 * aud["with_auditd_main_process"] / aud["cells"]
_aud_other = lc[(lc["variant"] != "freerasp_debug") & lc["has_auditd"]]
assert set(_aud_other["app"]) <= {"documenter"} and not _aud_other["has_auditd_main"].any(), \
    "a denial outside freeRASP that is not documenter's worker"
aud["cells_with_denials_outside_documenter"] = (lc[lc["has_auditd"] & (lc["app"] != "documenter")]
                                               .groupby("variant").size().reindex(aud.index).fillna(0).astype(int))
aud.index = aud.index.map(T.name)
aud.index.name = "technique"
aud.to_csv(T.TABLES / "auditd_discriminator.csv")
T.register("tables/auditd_discriminator.csv", "REPO", "6.4 RQ3 (mechanism)",
           "analysed cells with SELinux denials per technique: from any of the application's processes, from its main process only, and outside documenter", NB)
print(aud.to_string())
non = aud.drop(index="freeRASP")
print(f"\nother builds: {int(non['with_auditd'].sum())} of {int(non['cells'].sum())} "
      f"(main process: {int(non['with_auditd_main_process'].sum())}; outside documenter: {int(non['cells_with_denials_outside_documenter'].sum())});  "
      f"freeRASP: {int(aud.loc['freeRASP','with_auditd'])} of {int(aud.loc['freeRASP','cells'])}")
_aud_dev = lc.groupby(["phone", "variant"]).agg(cells=("has_auditd", "size"), with_auditd=("has_auditd", "sum"),
                                                with_auditd_main_process=("has_auditd_main", "sum")).reset_index()
_aud_dev["device"] = _aud_dev["phone"].map(T.PHONE_DISPLAY)
_aud_dev["technique"] = _aud_dev["variant"].map(T.name)
_aud_dev = _aud_dev[["device", "technique", "cells", "with_auditd", "with_auditd_main_process"]]
assert (_aud_dev.groupby("technique")[["cells", "with_auditd", "with_auditd_main_process"]].sum()
        .reindex(aud.index)[["cells", "with_auditd", "with_auditd_main_process"]] == aud[["cells", "with_auditd", "with_auditd_main_process"]]).all().all()
_aud_dev.to_csv(T.TABLES / "auditd_per_device.csv", index=False)
T.register("tables/auditd_per_device.csv", "REPO", "Supplement (runtime shielding per device)",
           "analysed cells with SELinux denials per device and technique: from any of the application's processes, and from its main process", NB)
print(_aud_dev.to_string(index=False))
T.record("mechanism.auditd", freerasp=f"{int(aud.loc['freeRASP','with_auditd'])} of {int(aud.loc['freeRASP','cells'])}",
         other_builds=f"{int(non['with_auditd'].sum())} of {int(non['cells'].sum())}",
         other_builds_main_process=f"{int(non['with_auditd_main_process'].sum())} of {int(non['cells'].sum())}",
         other_builds_source="documenter's conversion worker (:pdf2htmlEXWorker), one denial per cell in every build",
         unit="analysed cells")


           cells  with_auditd  with_auditd_main_process   share_pct  share_main_process_pct  cells_with_denials_outside_documenter
technique                                                                                                                         
Allatori     439            0                         0    0.000000                     0.0                                      0
Bangcle      676           90                         0   13.313609                     0.0                                      0
Control      760           89                         0   11.710526                     0.0                                      0
freeRASP     770          770                       770  100.000000                   100.0                                    682

other builds: 179 of 1875 (main process: 0; outside documenter: 0);  freeRASP: 770 of 770
 device technique  cells  with_auditd  with_auditd_main_process
Pixel 3  Allatori    139            0                      

## §6.4 (RQ3): Thread counts

**What `threads` counts:** the distinct threads (process ID, thread ID) that wrote at least one line to the log during the
cell (`tools/logcat_app_parser.py`). A thread that never logs is not seen, and a process started twice in one cell
contributes its threads twice. So this is *logging threads*, not threads alive at one moment.

Matched within (application, device): mean threads of each technique's cells against the same application's control cells on
the same device. Quoted as an absolute per-application range, which does not depend on an averaging choice the way a pooled
percentage does.

In [5]:
g = lc.groupby(["app", "phone", "variant"])["threads"].mean().unstack("variant")
rows = []
for arm in T.ARMS:
    d = (g[arm] - g["baseline_debug"]).dropna()
    per_app = d.groupby(level="app").mean()
    for app, v in per_app.items():
        rows.append(dict(app=app, technique=T.name(arm), delta_threads=v,
                         control_threads=g["baseline_debug"].xs(app, level="app").mean()))
th = pd.DataFrame(rows); th.to_csv(T.TABLES / "thread_deltas_per_app.csv", index=False)
T.register("tables/thread_deltas_per_app.csv", "REPO", "6.4 RQ3 (mechanism)", "change in thread count vs control, per application and technique", NB)
print(th.pivot(index="app", columns="technique", values="delta_threads").round(2).to_string())
f = th[th["technique"] == "freeRASP"]
print(f"\nfreeRASP: +{f.delta_threads.min():.1f} to +{f.delta_threads.max():.1f} threads per app, "
      f"on controls of {f.control_threads.min():.0f} to {f.control_threads.max():.0f}; positive in {(f.delta_threads>0).sum()} of {len(f)}")
T.record("mechanism.freerasp_threads", range=[f.delta_threads.min(), f.delta_threads.max()],
         positive=f"{int((f.delta_threads > 0).sum())} of {len(f)}")

technique     Allatori  Bangcle  freeRASP
app                                      
calculator        0.09    -0.83      4.47
documenter         NaN     0.27      7.90
itsok              NaN     0.29      5.50
keep_recipe        NaN    -0.08      6.15
linkhub            NaN     0.02      5.53
medtimer          0.07      NaN      5.62
pdfviewer        -0.27    -0.19      5.07
poetskingdom     -0.59     0.36     13.81
tipuous          -0.15     0.43      6.79

freeRASP: +4.5 to +13.8 threads per app, on controls of 9 to 26; positive in 9 of 9


## §6.4 (RQ3) / Discussion: When freeRASP's SELinux denials happen

The denials counted above, split by time (`denial_timing_series.csv`, `tools/extract_denial_timing.py`; each cell's split
sums to the parser's count, which the extractor checks): before energy measurement (the launch and the post-launch hook), in
the post-launch window, in the workload window, and after it. Analysed freeRASP cells only: the other builds' denials
are documenter's worker's, one per cell (above).
Rates are pooled: the window's denials over the window's total minutes. Exploratory.

In [6]:
_dt_all = pd.read_csv(T.DATA / "inputs" / "denial_timing_series.csv", dtype={"rep": str}).merge(series[key], on=key, how="inner")
assert set(_dt_all.loc[_dt_all["variant"] != "freerasp_debug", "app"]) <= {"documenter"}, "a denial outside freeRASP and documenter"
_dt = _dt_all[_dt_all["variant"] == "freerasp_debug"].copy()
assert len(_dt) == int(aud.loc["freeRASP", "with_auditd"]), "a freeRASP cell with denials is missing from the timing input"
_dt_total = int(_dt["auditd_lines"].sum())
_dt_rows = []
# labelled groups: the passive-phase analysis leaves out medtimer, whose passive phase is ~100 s
for _dt_grp, _dt_g in (("the eight other than medtimer", _dt[~_dt["app"].isin(T.PASSIVE_EXCLUDED_APPS)]),
                       ("all nine (medtimer included)", _dt), ("medtimer", _dt[_dt["app"].isin(T.PASSIVE_EXCLUDED_APPS)])):
    if _dt_g.empty:
        continue
    _dt_gt = int(_dt_g["auditd_lines"].sum())
    for _dt_w, _dt_len in (("before", None), ("post_launch", "post_launch_s"), ("workload", "workload_s"), ("after", None)):
        _dt_rows.append({"applications": _dt_grp, "window": {"before": "before measurement (launch)", "post_launch": "passive phase",
                                                         "workload": "active phase", "after": "after the active phase"}[_dt_w],
                         "cells": len(_dt_g), "denials": int(_dt_g[_dt_w].sum()), "share_pct": 100 * _dt_g[_dt_w].sum() / _dt_gt,
                         "cells_with_any": int((_dt_g[_dt_w] > 0).sum()), "median_per_cell": _dt_g[_dt_w].median(),
                         "per_minute_pooled": (_dt_g[_dt_w].sum() / (_dt_g[_dt_len].sum() / 60)) if _dt_len else float("nan")})
_dt_tab = pd.DataFrame(_dt_rows)
assert _dt_tab.loc[_dt_tab["applications"] == "all nine (medtimer included)", "denials"].sum() == _dt_total
_dt_tab.to_csv(T.TABLES / "selinux_denial_timing.csv", index=False)
T.register("tables/selinux_denial_timing.csv", "REPO", "6.4 RQ3 / Discussion (when the denials happen)",
           "freeRASP's SELinux denials by time: before measurement, passive phase, active phase, after; share and rate per minute; "
           "the eight applications of the passive analysis, all nine, and medtimer", NB)
print(_dt_tab.round(2).to_string(index=False))
_dt_8 = _dt_tab[_dt_tab["applications"] == "the eight other than medtimer"].set_index("window")
T.record("mechanism.denial_timing", share_pct=_dt_8["share_pct"].to_dict(),
         per_minute_post_launch=_dt_8.loc["passive phase", "per_minute_pooled"], per_minute_workload=_dt_8.loc["active phase", "per_minute_pooled"],
         cells=int(_dt_8["cells"].iloc[0]),
         basis="analysed freeRASP cells with denials, without medtimer; zero = the cell's first energy sample")

                 applications                      window  cells  denials  share_pct  cells_with_any  median_per_cell  per_minute_pooled
the eight other than medtimer before measurement (launch)    680     5864      26.62             679              8.0                NaN
the eight other than medtimer               passive phase    680     1751       7.95             260              0.0               5.09
the eight other than medtimer                active phase    680    13453      61.06             680             18.0              10.41
the eight other than medtimer      after the active phase    680      964       4.38             198              0.0                NaN
 all nine (medtimer included) before measurement (launch)    770     6742      26.68             769              8.0                NaN
 all nine (medtimer included)               passive phase    770     2838      11.23             350              0.0               5.74
 all nine (medtimer included)            

## §6.4 (RQ3): Where freeRASP's post-launch cost goes (exploratory)

**Not pre-specified.** Found after the fact, from channels the design did not plan to analyse: power rails,
other processes' logs, and traffic counters. Input: `mechanism_cells_series.csv`, with `mechanism_data_alerts_series.csv`,
`mechanism_local_server_series.csv` and `proxy_connections.csv` (`tools/extract_freerasp_mechanism.py`;
`freerasp_mechanism.PROVENANCE.md`).
- **Cells:** analysed series cells, as everywhere.
- **Statistics:** matched pairs (application × device × batch, each the mean over its cells) and paired t-intervals.
- **Time zero:** the cell's first energy sample.
- **Segments:** the boundaries at 4 s and 15 s were chosen after inspecting the per-second profile below.

What the channels are:
- **Rails:** the On-Device Power Monitor readings the Pixel 6 and 9 thermal service logs every 7 s. The Pixel 3 has no
  such log.
- **Local-server rejections:** the YouTube and YouTube Music apps' local web servers logging a request they rejected.
- **Polls:** Android reading its traffic counters at its data-usage alert, about every 2 MB of traffic.

freeRASP is closed source: what it does is inferred from these side effects, not read from its code.


In [7]:
_fm = pd.read_csv(T.DATA / "inputs" / "mechanism_cells_series.csv", dtype={"rep": str}).merge(series[key], on=key, how="inner")
_FM_ARMS = ("freerasp_debug", "bangcle_debug", "allatori_debug")
_FM_SEG = {"0_4": "0-4 s", "4_15": "4-15 s (the burst)", "15_end": "15 s to the end of the phase",
           "post_launch": "whole passive phase", "workload": "active phase"}
_FM_DEV = {"pixel3": "Pixel 3", "pixel6": "Pixel 6", "pixel9": "Pixel 9"}


def _fm_pairs(d, col, arm):
    """Matched pairs: the arm's mean over its cells minus the control's, per application x device x batch."""
    g = d.dropna(subset=[col]).groupby(["app", "phone", "batch", "variant"])[col].mean().unstack("variant")
    if arm not in g or "baseline_debug" not in g:
        return pd.Series(dtype=float), pd.Series(dtype=float)
    ok = g[arm].notna() & g["baseline_debug"].notna()
    return g.loc[ok, arm] - g.loc[ok, "baseline_debug"], g.loc[ok, "baseline_debug"]


def _fm_row(d, col, arm, **label):
    x, base = _fm_pairs(d, col, arm)
    if len(x) < 2:
        return dict(**label, n_pairs=len(x))
    m, lo, hi, _ = T.ci_naive(x)
    return dict(**label, n_pairs=len(x), control_mean=base.mean(), delta=m, ci_lo=lo, ci_hi=hi,
                pairs_higher_pct=100 * (x > 0).mean())


# the extractor's window energies against the run table's (the same windows, the same samples)
_fm_chk = _fm.merge(series[key + ["pre_workload_joules", "workload_joules"]], on=key)
_fm_dev_pl = ((_fm_chk["e_post_launch_j"] - _fm_chk["pre_workload_joules"]).abs() / _fm_chk["pre_workload_joules"]).median()
_fm_dev_wl = ((_fm_chk["e_workload_j"] - _fm_chk["workload_joules"]).abs() / _fm_chk["workload_joules"]).median()
assert _fm_dev_pl < 0.005 and _fm_dev_wl < 0.005, (_fm_dev_pl, _fm_dev_wl)
print(f"{len(_fm)} analysed cells, {int(_fm['has_log'].sum())} with a log; window energies against the run table: "
      f"median |difference| {100 * _fm_dev_pl:.2f}% (passive-phase), {100 * _fm_dev_wl:.2f}% (workload)")


2647 analysed cells, 2645 with a log; window energies against the run table: median |difference| 0.13% (passive-phase), 0.09% (workload)


### Energy by segment of the post-launch window

`freerasp_postlaunch_segments.csv`: extra energy per cell, protected minus control, by segment. For freeRASP, the
segments are reported for all applications, the eight with the INTERNET permission, each application, and each device.


In [8]:
_fm_rows = []
for _fm_arm in _FM_ARMS:
    _fm_groups = [("all applications", _fm)]
    if _fm_arm == "freerasp_debug":
        _fm_groups += [("the eight other than medtimer", _fm[_fm["app"] != "medtimer"])]
        _fm_groups += [(a, _fm[_fm["app"] == a]) for a in sorted(_fm["app"].unique())]
        _fm_groups += [(_FM_DEV[p], _fm[_fm["phone"] == p]) for p in _FM_DEV]
    for _fm_g, _fm_d in _fm_groups:
        for _fm_s, _fm_lab in _FM_SEG.items():
            # the passive segments use the passive-phase set (no medtimer), except in medtimer's own row
            _fm_src = _fm_d if (_fm_s == "workload" or _fm_g == "medtimer") else T.phase_set(_fm_d, "passive")
            _fm_rows.append(_fm_row(_fm_src, f"e_{_fm_s}_j", _fm_arm, technique=T.name(_fm_arm), group=_fm_g, segment=_fm_lab))
_fm_segtab = pd.DataFrame(_fm_rows)
_fm_segtab.to_csv(T.TABLES / "freerasp_postlaunch_segments.csv", index=False)
T.register("tables/freerasp_postlaunch_segments.csv", "REPO", "6.4 RQ3 (where the passive-phase cost goes; exploratory)",
           "extra energy per cell by segment of the passive phase (0-4 s, 4-15 s, the rest) and in the active phase, per technique; freeRASP also per application and device; "
           "the passive segments without medtimer except in its own row", NB)
_fm_s = _fm_segtab[_fm_segtab["technique"] == "freeRASP"].pivot(index="group", columns="segment", values="delta")[list(_FM_SEG.values())]
print(_fm_s.round(2).to_string())
_fm_all = _fm_segtab[(_fm_segtab["technique"] == "freeRASP") & (_fm_segtab["group"] == "all applications")].set_index("segment")
_fm_burst_share = 100 * _fm_all.loc[_FM_SEG["4_15"], "delta"] / _fm_all.loc[_FM_SEG["post_launch"], "delta"]
print(f"\nthe burst segment is {_fm_burst_share:.1f}% of freeRASP's extra passive-phase energy")
print(_fm_segtab[(_fm_segtab["technique"] != "freeRASP")].round(2).to_string(index=False))


segment                        0-4 s  4-15 s (the burst)  15 s to the end of the phase  whole passive phase  active phase
group                                                                                                                    
Pixel 3                         2.82                8.85                          7.34                19.02         19.77
Pixel 6                         0.86               29.73                          0.01                30.59         12.76
Pixel 9                         1.49               15.93                         -0.51                16.90          0.18
all applications                1.67               18.57                          2.06                22.31         10.56
calculator                      1.96               19.26                         -0.41                20.81          8.67
documenter                      2.37               18.33                          3.69                24.39         22.84
itsok                   

### Power second by second

`freerasp_postlaunch_profile.csv`: extra power in each second of the post-launch window, per technique and device.
`freerasp_burst_timing.csv`: for each device, the longest run of consecutive seconds in which freeRASP's extra power is at
least 1.0 W (the core) and at least 0.5 W (the extent), each with its interval above zero.


In [9]:
_fm_rows = []
_fm_pas = _fm[~_fm["app"].isin(T.PASSIVE_EXCLUDED_APPS)]          # the passive-phase analysis set (no medtimer)
for _fm_arm in _FM_ARMS:
    for _fm_p in ["all"] + list(_FM_DEV):
        _fm_d = _fm_pas if _fm_p == "all" else _fm_pas[_fm_pas["phone"] == _fm_p]
        for _fm_k in range(30):
            _fm_rows.append(_fm_row(_fm_d, f"p_{_fm_k:02d}", _fm_arm, technique=T.name(_fm_arm),
                                    device="all devices" if _fm_p == "all" else _FM_DEV[_fm_p],
                                    second_from=_fm_k, second_to=_fm_k + 1))
    for _fm_lab, _fm_d in (("all devices, medtimer only", _fm[_fm["app"].isin(T.PASSIVE_EXCLUDED_APPS)]),
                           ("all devices, all nine applications", _fm)):
        for _fm_k in range(30):
            _fm_rows.append(_fm_row(_fm_d, f"p_{_fm_k:02d}", _fm_arm, technique=T.name(_fm_arm), device=_fm_lab,
                                    second_from=_fm_k, second_to=_fm_k + 1))
_fm_prof = pd.DataFrame(_fm_rows)
_fm_prof.to_csv(T.TABLES / "freerasp_postlaunch_profile.csv", index=False)
T.register("tables/freerasp_postlaunch_profile.csv", "REPO", "6.4 RQ3 (where the passive-phase cost goes; exploratory)",
           "extra power in each second of the passive phase, per technique and device, with interval and pair count", NB)


def _fm_longest(g, threshold):
    on = ((g["delta"] >= threshold) & (g["ci_lo"] > 0)).tolist()
    best, cur, start = (None, None, 0), 0, None
    for i, v in enumerate(on):
        cur = cur + 1 if v else 0
        if v and cur == 1:
            start = i
        if cur > best[2]:
            best = (int(g["second_from"].iloc[start]), int(g["second_to"].iloc[i]), cur)
    return best


_fm_rows = []
for _fm_dev, _fm_g in _fm_prof[(_fm_prof["technique"] == "freeRASP") & _fm_prof["device"].isin(["all devices", *_FM_DEV.values()])].groupby("device", sort=False):
    _fm_g = _fm_g.sort_values("second_from").reset_index(drop=True)
    c0, c1, cn = _fm_longest(_fm_g, 1.0)
    e0, e1, en = _fm_longest(_fm_g, 0.5)
    pk = _fm_g.loc[_fm_g["delta"].idxmax()]
    _fm_rows.append(dict(device=_fm_dev, core_from_s=c0, core_to_s=c1, extent_from_s=e0, extent_to_s=e1,
                         peak_delta_w=pk["delta"], peak_second_from=int(pk["second_from"]), n_pairs=int(_fm_g["n_pairs"].max())))
_fm_btiming = pd.DataFrame(_fm_rows)
_fm_btiming.to_csv(T.TABLES / "freerasp_burst_timing.csv", index=False)
T.register("tables/freerasp_burst_timing.csv", "REPO", "6.4 RQ3 (where the passive-phase cost goes; exploratory)",
           "when freeRASP's passive-phase burst starts and ends, per device: longest runs of seconds with extra power >= 1.0 W and >= 0.5 W, peak", NB)
print(_fm_btiming.round(2).to_string(index=False))
print(_fm_prof[(_fm_prof["technique"] == "freeRASP") & (_fm_prof["device"] == "all devices")][
    ["second_from", "n_pairs", "control_mean", "delta", "ci_lo", "ci_hi", "pairs_higher_pct"]].round(2).to_string(index=False))


     device  core_from_s  core_to_s  extent_from_s  extent_to_s  peak_delta_w  peak_second_from  n_pairs
all devices            4         15              4           19          2.02                 9       69
    Pixel 3           11         21              8           30          1.28                13       21
    Pixel 6            4         15              4           16          3.33                 6       24
    Pixel 9            4         12              3           16          2.11                 8       24
 second_from  n_pairs  control_mean  delta  ci_lo  ci_hi  pairs_higher_pct
           0       69          2.90   0.53   0.42   0.64             88.41
           1       69          2.88   0.40   0.29   0.50             76.81
           2       69          3.15   0.31   0.23   0.38             85.51
           3       69          3.08   0.46   0.32   0.59             82.61
           4       69          2.70   1.29   1.04   1.53             97.10
           5       69    

### What kind of work: the CPU rails (Pixel 6 and 9)

`freerasp_cpu_rails.csv`: extra power on the battery and on each rail, per window, per technique; the two devices
together and each alone. The windows run from 8 s before measurement to the end of the workload; `burst_4_15` and
`after_burst` split the post-launch window. Android Runner's own CPU figure (`dumpsys cpuinfo`, a share of time over
all cores, about one sample per 1.3 s) is in the same table, on all three devices, for comparison.


In [10]:
_FM_RW = {"before": "8 s before measurement", "post_launch": "passive phase", "burst_4_15": "passive phase, 4-15 s",
          "after_burst": "passive phase, after 15 s", "workload": "active phase"}
_FM_RM = {"battery": "battery", "cpu_total": "all CPU rails", "cpu_big": "big cluster", "cpu_mid": "middle cluster",
          "cpu_little": "little cluster", "gpu": "GPU"}
_fm_rows = []
for _fm_arm in _FM_ARMS:
    for _fm_p, _fm_d in (("Pixel 6 and 9", _fm[_fm["phone"].isin(["pixel6", "pixel9"])]),
                         ("Pixel 6", _fm[_fm["phone"] == "pixel6"]), ("Pixel 9", _fm[_fm["phone"] == "pixel9"])):
        for _fm_w, _fm_wl in _FM_RW.items():
            _fm_src = _fm_d if _fm_w in ("before", "workload") else T.phase_set(_fm_d, "passive")   # passive windows: no medtimer
            for _fm_m, _fm_ml in _FM_RM.items():
                _fm_col = f"rail_{_fm_w}_{_fm_m}"
                if _fm_col in _fm_src:
                    _fm_rows.append(_fm_row(_fm_src, _fm_col, _fm_arm, technique=T.name(_fm_arm), devices=_fm_p,
                                            window=_fm_wl, measure=_fm_ml, unit="W"))
    for _fm_p, _fm_d in (("all three", _fm), ("Pixel 6 and 9", _fm[_fm["phone"].isin(["pixel6", "pixel9"])])):
        for _fm_w in ("post_launch", "workload"):
            _fm_rows.append(_fm_row(_fm_d if _fm_w == "workload" else T.phase_set(_fm_d, "passive"), f"ar_cpu_{_fm_w}", _fm_arm,
                                    technique=T.name(_fm_arm), devices=_fm_p,
                                    window=_FM_RW[_fm_w], measure="Android Runner CPU (share of time)", unit="% points"))
_fm_rails = pd.DataFrame(_fm_rows)
_fm_rails["note"] = "the Pixel 3 has no rail log; the passive windows leave out medtimer"
_fm_rails.to_csv(T.TABLES / "freerasp_cpu_rails.csv", index=False)
T.register("tables/freerasp_cpu_rails.csv", "REPO", "6.4 RQ3 (where the passive-phase cost goes; exploratory)",
           "extra power on the battery and on each CPU cluster and the GPU (Pixel 6 and 9 rail log), per window and technique; Android Runner's CPU share beside it", NB)
_fm_r = _fm_rails[(_fm_rails["technique"] == "freeRASP") & (_fm_rails["devices"] == "Pixel 6 and 9")].set_index(["window", "measure"])
print(_fm_r[["n_pairs", "control_mean", "delta", "ci_lo", "ci_hi", "pairs_higher_pct"]].round(3).to_string())
_fm_cpu_share_pl = 100 * _fm_r.loc[("passive phase", "all CPU rails"), "delta"] / _fm_r.loc[("passive phase", "battery"), "delta"]
_fm_cpu_share_b = 100 * _fm_r.loc[("passive phase, 4-15 s", "all CPU rails"), "delta"] / _fm_r.loc[("passive phase, 4-15 s", "battery"), "delta"]
print(f"\nCPU rails' share of the extra battery power: passive-phase {_fm_cpu_share_pl:.0f}%, 4-15 s {_fm_cpu_share_b:.0f}%")
print(_fm_rails[(_fm_rails["technique"] != "freeRASP") & (_fm_rails["devices"] == "Pixel 6 and 9") & _fm_rails["window"].isin(["passive phase", "passive phase, 4-15 s"])
                & _fm_rails["measure"].isin(["battery", "all CPU rails"])][["technique", "window", "measure", "n_pairs", "delta", "ci_lo", "ci_hi"]].round(3).to_string(index=False))


                                                              n_pairs  control_mean  delta  ci_lo  ci_hi  pairs_higher_pct
window                    measure                                                                                         
8 s before measurement    all CPU rails                            54         0.569  0.790  0.740  0.840           100.000
                          big cluster                              54         0.252  0.574  0.530  0.617           100.000
                          middle cluster                           54         0.149  0.124  0.108  0.139           100.000
                          little cluster                           54         0.168  0.093  0.084  0.101           100.000
                          GPU                                      54         0.011  0.001  0.000  0.001            66.667
passive phase             battery                                  48         2.782  0.903  0.779  1.026            97.917
                

### What freeRASP does in the burst: other applications' local servers receive requests

`freerasp_local_probes.csv`: per application and device, the requests the YouTube and YouTube Music apps' local web
servers rejected ("SignatureInterceptor: URL not signed correctly"), per cell, in freeRASP and control cells; with
`freerasp_local_probe_timing.csv`, when they happen.


In [11]:
_fm_log = _fm[_fm["has_log"] == 1]
_fm_rows = []
for _fm_p in ["all devices"] + list(_FM_DEV):
    _fm_d = _fm_log if _fm_p == "all devices" else _fm_log[_fm_log["phone"] == _fm_p]
    for _fm_a in ["the eight other than medtimer", "all applications (medtimer included)"] + sorted(_fm_log["app"].unique()):
        _fm_x = (_fm_d[~_fm_d["app"].isin(T.PASSIVE_EXCLUDED_APPS)] if _fm_a.startswith("the eight")
                 else _fm_d if _fm_a.startswith("all applications") else _fm_d[_fm_d["app"] == _fm_a])
        for _fm_v in ("freerasp_debug", "baseline_debug", "bangcle_debug", "allatori_debug"):
            _fm_c = _fm_x[_fm_x["variant"] == _fm_v]
            if len(_fm_c):
                _fm_rows.append(dict(device=_FM_DEV.get(_fm_p, _fm_p), application=_fm_a, build=T.name(_fm_v), cells=len(_fm_c),
                                     per_cell_post_launch=_fm_c["local_server_hit_post_launch"].mean(),
                                     per_cell_workload=_fm_c["local_server_hit_workload"].mean(),
                                     cells_with_any_post_launch_pct=100 * (_fm_c["local_server_hit_post_launch"] > 0).mean()))
_fm_lp = pd.DataFrame(_fm_rows)
_fm_lp.to_csv(T.TABLES / "freerasp_local_probes.csv", index=False)
_fm_ht = pd.read_csv(T.DATA / "inputs" / "mechanism_local_server_series.csv", dtype={"rep": str}).merge(series[key], on=key)
_fm_rows = []
for _fm_p, _fm_g in _fm_ht[_fm_ht["variant"] == "freerasp_debug"].groupby("phone"):
    _fm_rows.append(dict(device=_FM_DEV[_fm_p], lines=len(_fm_g), in_post_launch=int((_fm_g["window"] == "post_launch").sum()),
                         in_workload=int((_fm_g["window"] == "workload").sum()),
                         **{f"post_launch_{q}_s": _fm_g.loc[_fm_g["window"] == "post_launch", "t_s"].quantile(v)
                            for q, v in (("p10", .1), ("p25", .25), ("median", .5), ("p75", .75), ("p90", .9))}))
_fm_lpt = pd.DataFrame(_fm_rows)
_fm_lpt.to_csv(T.TABLES / "freerasp_local_probe_timing.csv", index=False)
assert _fm_ht[_fm_ht["variant"] != "freerasp_debug"].empty, "a rejected local request outside freeRASP"
T.register(["tables/freerasp_local_probes.csv", "tables/freerasp_local_probe_timing.csv"], "REPO",
           "6.4 RQ3 (where the passive-phase cost goes; exploratory)",
           "requests rejected by other applications' local servers, per cell, per application, device and build; when they happen, per device", NB)
print(_fm_lp[(_fm_lp["device"] == "all devices") & (_fm_lp["application"] == "the eight other than medtimer")].round(2).to_string(index=False))
print(_fm_lp[(_fm_lp["device"] == "all devices") & (_fm_lp["build"] == "freeRASP")].round(2).to_string(index=False))
print(_fm_lp[(_fm_lp["application"] == "the eight other than medtimer") & (_fm_lp["build"] == "freeRASP")].round(2).to_string(index=False))
print(_fm_lpt.round(1).to_string(index=False))


     device                   application    build  cells  per_cell_post_launch  per_cell_workload  cells_with_any_post_launch_pct
all devices the eight other than medtimer freeRASP    680                  1.02               0.13                           68.38
all devices the eight other than medtimer  Control    673                  0.00               0.00                            0.00
all devices the eight other than medtimer  Bangcle    676                  0.00               0.00                            0.00
all devices the eight other than medtimer Allatori    350                  0.00               0.00                            0.00
     device                          application    build  cells  per_cell_post_launch  per_cell_workload  cells_with_any_post_launch_pct
all devices        the eight other than medtimer freeRASP    680                  1.02               0.13                           68.38
all devices all applications (medtimer included) freeRASP    770     

### Traffic the phone sends to itself: the data-usage alert

`freerasp_data_alerts.csv`: the Wi-Fi traffic between two consecutive polls of the traffic counters, and the time between
them, per device, build, window (that of the later poll; `post-launch` excludes 4-15 s), for medtimer and for the other
eight. `freerasp_data_alert_polls.csv`: extra polls per cell in the post-launch window, per application.

How to read it:
- **In the control,** a poll follows about 2 MB of Wi-Fi traffic.
- **A poll with little Wi-Fi traffic behind it** was set off by traffic on another interface. The mobile counters are
  zero, so that interface is loopback, i.e. the phone itself. That reading is an inference: it assumes the alert counts
  all interfaces.


In [12]:
_fm_al = pd.read_csv(T.DATA / "inputs" / "mechanism_data_alerts_series.csv", dtype={"rep": str}).merge(series[key], on=key)
_fm_al = _fm_al[_fm_al["variant"].isin(["baseline_debug", "freerasp_debug"]) & _fm_al["window"].isin(["burst_4_15", "post_launch", "workload"])]
_fm_al["applications"] = np.where(_fm_al["app"] == "medtimer", "medtimer", "the other eight")
_fm_al_tab = (_fm_al.groupby(["phone", "applications", "window", "variant"])
              .agg(intervals=("wifi_mb", "size"), median_wifi_mb=("wifi_mb", "median"),
                   p10_wifi_mb=("wifi_mb", lambda x: x.quantile(.1)), p90_wifi_mb=("wifi_mb", lambda x: x.quantile(.9)),
                   median_gap_s=("gap_s", "median")).reset_index())
_fm_al_tab["phone"] = _fm_al_tab["phone"].map(_FM_DEV)
_fm_al_tab["variant"] = _fm_al_tab["variant"].map(T.name)
_fm_al_tab["window"] = _fm_al_tab["window"].map({"burst_4_15": "passive phase, 4-15 s", "post_launch": "passive phase, other seconds", "workload": "active phase"})
_fm_al_tab = _fm_al_tab.rename(columns={"phone": "device", "variant": "build"})
_fm_al_tab.to_csv(T.TABLES / "freerasp_data_alerts.csv", index=False)
_fm_rows = []
for _fm_a in ["all applications", "the eight other than medtimer"] + sorted(_fm_log["app"].unique()):
    _fm_d = (_fm_log if _fm_a == "all applications" else _fm_log[_fm_log["app"] != "medtimer"] if _fm_a.startswith("the eight")
             else _fm_log[_fm_log["app"] == _fm_a])
    _fm_rows.append(_fm_row(_fm_d, "data_alert_poll_post_launch", "freerasp_debug", application=_fm_a, unit="polls per cell"))
_fm_polls = pd.DataFrame(_fm_rows)
_fm_polls.to_csv(T.TABLES / "freerasp_data_alert_polls.csv", index=False)
T.register(["tables/freerasp_data_alerts.csv", "tables/freerasp_data_alert_polls.csv"], "REPO",
           "6.4 RQ3 (where the passive-phase cost goes; exploratory)",
           "Wi-Fi traffic between consecutive polls of the traffic counters, and the time between them, per device, build and window; extra polls per cell in the passive phase", NB)
print(_fm_al_tab.round(2).to_string(index=False))
print(_fm_polls.round(2).to_string(index=False))


 device    applications                       window    build  intervals  median_wifi_mb  p10_wifi_mb  p90_wifi_mb  median_gap_s
Pixel 3        medtimer        passive phase, 4-15 s  Control         19            2.06         2.02         2.17         35.36
Pixel 3        medtimer        passive phase, 4-15 s freeRASP         27            2.03         2.01         2.06         34.13
Pixel 3        medtimer passive phase, other seconds  Control         94            1.65         1.63         1.70         25.90
Pixel 3        medtimer passive phase, other seconds freeRASP         93            1.65         1.64         1.67         25.52
Pixel 3        medtimer                 active phase  Control        121            2.12         1.98         2.19         24.66
Pixel 3        medtimer                 active phase freeRASP        123            2.12         2.02         2.18         23.61
Pixel 3 the other eight        passive phase, 4-15 s  Control         95            2.00         

### Not reporting home: Wi-Fi traffic over the cell, and the proxy logs

`freerasp_wifi_rate.csv`: Wi-Fi traffic over the measured cell (from the first to the last poll inside it), freeRASP minus
control, per device. `freerasp_proxy_check.csv`: the four sweeps' mitmproxy logs. The proxy was set only for pdfviewer's
cells (the one application with recorded network flows; `tools/sweep_runner_parallel_ykush.sh`), so those logs speak for
pdfviewer only.


In [13]:
_fm_rows = [_fm_row(_fm_log if _fm_p == "all devices" else _fm_log[_fm_log["phone"] == _fm_p], "wifi_kb_per_s", "freerasp_debug",
                   device=_FM_DEV.get(_fm_p, _fm_p), unit="kB/s") for _fm_p in ["all devices"] + list(_FM_DEV)]
_fm_wifi = pd.DataFrame(_fm_rows)
_fm_wifi.to_csv(T.TABLES / "freerasp_wifi_rate.csv", index=False)
_fm_px = pd.read_csv(T.DATA / "inputs" / "proxy_connections.csv")
_fm_px["scope"] = "pdfviewer's cells only: the proxy was set for no other application"
_fm_px.to_csv(T.TABLES / "freerasp_proxy_check.csv", index=False)
T.register(["tables/freerasp_wifi_rate.csv", "tables/freerasp_proxy_check.csv"], "REPO",
           "6.4 RQ3 (where the passive-phase cost goes; exploratory)",
           "Wi-Fi traffic over the measured cell, freeRASP minus control, per device; the four sweeps' proxy logs: connections and any to Talsec (pdfviewer cells only)", NB)
print(_fm_wifi.round(2).to_string(index=False))
print(_fm_px[["sweep", "client_connections", "connections_to_talsec", "lines_naming_talsec"]].to_string(index=False))


     device unit  n_pairs  control_mean  delta  ci_lo  ci_hi  pairs_higher_pct
all devices kB/s       78        159.32   0.08  -1.03   1.19             66.67
    Pixel 3 kB/s       24        126.30   0.13  -2.52   2.78             70.83
    Pixel 6 kB/s       27        179.05  -1.17  -2.69   0.35             59.26
    Pixel 9 kB/s       27        168.94   1.28  -0.43   3.00             70.37
  sweep  client_connections  connections_to_talsec  lines_naming_talsec
  pilot                 103                      0                    0
series1                 287                      0                    0
series2                 883                      0                    0
series3                 910                      0                    0


### The other checks, visible only in other processes' logs

`freerasp_system_traces.csv`: lines per cell, freeRASP and control, before measurement and in the post-launch window.
Every trace here is a line written by a process other than the application, about the application's requests, except
the last two rows. freeRASP's passive listener logs only a detection, so a check that finds nothing writes nothing of
its own.

Which devices log which traces:
- **Pixel 3:** no package-visibility lines and no package-list warnings.
- **Pixel 6 and 9:** no DRM or GL lines.


In [14]:
_FM_TR = {"pkg_list_reply": "oversized package-list replies (system_server)", "appsfilter": "package-visibility lines (system_server, AppsFilter)",
          "biometric_query": "biometric queries naming the application (BiometricService)", "keystore2": "keystore2 lines",
          "local_server_hit": "requests rejected by local servers (YouTube, YouTube Music)", "data_alert_poll": "polls of the traffic counters",
          "drm": "DRM lines (Pixel 3)", "gl_context": "GL-context lines (Pixel 3)",
          "app_denial": "SELinux denials (the application's processes)", "app_freerasp_tag": "freeRASP's own tags (the application's processes)"}
_fm_rows = []
for _fm_t, _fm_tl in _FM_TR.items():
    _fm_trow = dict(trace=_fm_tl)
    for _fm_v, _fm_vl in (("freerasp_debug", "freeRASP"), ("baseline_debug", "control")):
        _fm_c = _fm_log[_fm_log["variant"] == _fm_v]
        _fm_cp = T.phase_set(_fm_c, "passive")          # the passive phase leaves out medtimer; the launch keeps it
        _fm_trow[f"{_fm_vl}_before_per_cell"] = _fm_c[f"{_fm_t}_before"].mean()
        _fm_trow[f"{_fm_vl}_post_launch_per_cell"] = _fm_cp[f"{_fm_t}_post_launch"].mean()
        _fm_trow[f"{_fm_vl}_post_launch_cells_with_any_pct"] = 100 * (_fm_cp[f"{_fm_t}_post_launch"] > 0).mean()
    _fm_rows.append(_fm_trow)
_fm_tr = pd.DataFrame(_fm_rows)
_fm_tr.to_csv(T.TABLES / "freerasp_system_traces.csv", index=False)
T.register("tables/freerasp_system_traces.csv", "REPO", "6.4 RQ3 (where the passive-phase cost goes; exploratory)",
           "traces of freeRASP's checks per cell, freeRASP and control, before measurement and in the passive phase; most are written by other processes", NB)
print(_fm_tr.round(2).to_string(index=False))


                                                      trace  freeRASP_before_per_cell  freeRASP_post_launch_per_cell  freeRASP_post_launch_cells_with_any_pct  control_before_per_cell  control_post_launch_per_cell  control_post_launch_cells_with_any_pct
             oversized package-list replies (system_server)                      0.16                           1.43                                    70.29                     0.16                          0.00                                    0.00
       package-visibility lines (system_server, AppsFilter)                    115.41                         234.17                                    70.29                     4.90                         12.76                                   70.88
biometric queries naming the application (BiometricService)                      5.28                           6.32                                   100.00                     0.00                          0.12                             

### medtimer, the one build that cannot open a network connection

`freerasp_medtimer_contrast.csv`: every measure above, for medtimer and for the other eight. medtimer's freeRASP build
lacks INTERNET: its manifest removes the permission, and the build kept the removal (`variant_parity_full.csv`). Without
INTERNET an application cannot open a socket at all, even to its own phone.

Each measure is reported two ways:
- as a pooled pair mean with its interval, for medtimer and for the eight together;
- as the lowest and highest of the eight applications' own means.


In [15]:
_fm_par = pd.read_csv(T.TABLES / "variant_parity_full.csv")
_fm_inet = _fm_par[_fm_par["variant"] == "freerasp_debug"].set_index("app")["declares_internet"]
assert not bool(_fm_inet["medtimer"]) and _fm_inet.drop("medtimer").astype(bool).all(), _fm_inet.to_dict()
_FM_MC = [("e_0_4_j", "extra energy, 0-4 s", "J", _fm), ("e_4_15_j", "extra energy, 4-15 s (the burst)", "J", _fm),
          ("e_15_end_j", "extra energy, 15 s to the end of the phase", "J", _fm), ("e_post_launch_j", "extra energy, whole passive phase", "J", _fm),
          ("e_workload_j", "extra energy, active phase", "J", _fm),
          ("rail_burst_4_15_battery", "extra battery power, 4-15 s (Pixel 6 and 9)", "W", _fm),
          ("rail_burst_4_15_cpu_total", "extra CPU-rail power, 4-15 s (Pixel 6 and 9)", "W", _fm),
          ("local_server_hit_post_launch", "requests rejected by local servers, passive phase, per cell", "count", _fm_log),
          ("data_alert_poll_post_launch", "polls of the traffic counters, passive phase, per cell", "count", _fm_log),
          ("post_launch_s", "length of the passive phase, freeRASP minus control", "s", _fm)]
_fm_rows = []
for _fm_col, _fm_lab, _fm_u, _fm_src in _FM_MC:
    _fm_m = _fm_row(_fm_src[_fm_src["app"] == "medtimer"], _fm_col, "freerasp_debug")
    _fm_e = _fm_row(_fm_src[_fm_src["app"] != "medtimer"], _fm_col, "freerasp_debug")
    _fm_apps = [_fm_pairs(_fm_src[_fm_src["app"] == a], _fm_col, "freerasp_debug")[0].mean() for a in sorted(set(_fm_src["app"]) - {"medtimer"})]
    _fm_rows.append(dict(measure=_fm_lab, unit=_fm_u, medtimer=_fm_m.get("delta"), medtimer_ci_lo=_fm_m.get("ci_lo"), medtimer_ci_hi=_fm_m.get("ci_hi"),
                         medtimer_pairs=_fm_m["n_pairs"], eight=_fm_e.get("delta"), eight_ci_lo=_fm_e.get("ci_lo"), eight_ci_hi=_fm_e.get("ci_hi"),
                         eight_pairs=_fm_e["n_pairs"], eight_lowest_app=np.nanmin(_fm_apps), eight_highest_app=np.nanmax(_fm_apps)))
_fm_fr = _fm[_fm["variant"] == "freerasp_debug"]
_fm_rows.append(dict(measure="length of the passive phase, freeRASP cells (mean)", unit="s",
                     medtimer=_fm_fr.loc[_fm_fr["app"] == "medtimer", "post_launch_s"].mean(),
                     eight=_fm_fr.loc[_fm_fr["app"] != "medtimer", "post_launch_s"].mean(),
                     eight_lowest_app=_fm_fr[_fm_fr["app"] != "medtimer"].groupby("app")["post_launch_s"].mean().min(),
                     eight_highest_app=_fm_fr[_fm_fr["app"] != "medtimer"].groupby("app")["post_launch_s"].mean().max()))
_fm_al8 = _fm_al_tab[(_fm_al_tab["window"] == "passive phase, 4-15 s") & (_fm_al_tab["build"] == "freeRASP")]
for _fm_p in ("Pixel 6", "Pixel 9"):
    _fm_q = _fm_al8[_fm_al8["device"] == _fm_p].set_index("applications")
    _fm_rows.append(dict(measure=f"Wi-Fi traffic between polls, 4-15 s, {_fm_p} (median, freeRASP cells)", unit="MB",
                         medtimer=_fm_q.loc["medtimer", "median_wifi_mb"], eight=_fm_q.loc["the other eight", "median_wifi_mb"]))
_fm_mc = pd.DataFrame(_fm_rows)
_fm_mc["freerasp_build_declares_internet"] = "medtimer: no; the eight: yes"
_fm_mc.to_csv(T.TABLES / "freerasp_medtimer_contrast.csv", index=False)
T.register("tables/freerasp_medtimer_contrast.csv", "REPO", "6.3 / 6.4 (medtimer; where the passive-phase cost goes)",
           "medtimer (freeRASP build without INTERNET) against the other eight on every measure of the passive-phase mechanism: pair means, intervals, the eight's range", NB)
print(_fm_mc.round(2).to_string(index=False))


                                                              measure  unit  medtimer  medtimer_ci_lo  medtimer_ci_hi  medtimer_pairs  eight  eight_ci_lo  eight_ci_hi  eight_pairs  eight_lowest_app  eight_highest_app freerasp_build_declares_internet
                                                  extra energy, 0-4 s     J      1.44            0.28            2.61             9.0   1.67         1.36         1.99         69.0              1.36               2.37     medtimer: no; the eight: yes
                                     extra energy, 4-15 s (the burst)     J      1.01           -0.43            2.46             9.0  18.57        16.32        20.83         69.0             17.00              23.22     medtimer: no; the eight: yes
                           extra energy, 15 s to the end of the phase     J      6.80            3.49           10.10             9.0   2.06         0.83         3.28         69.0             -0.41               3.69     medtimer: no; the eight: yes


### The instrument's own load: collections forced by the memory sampler (Threats)

Android Runner samples memory with `dumpsys meminfo <app>`, and the application runs an explicit garbage collection for
each call. `profiler_gc_load.csv` has two parts:
- **The collections**, per technique, as pairs: their rate, their duration per second, and the heap left after them.
- **A bound on their energy.** The extra duration is multiplied by the highest power each Pixel 6 and 9 CPU cluster was
  seen to draw: one 7-s reading, over all analysed cells. The product is set against freeRASP's workload power difference
  (`phase_durations_and_power.csv`).

It is a ceiling only at the rail log's resolution: it assumes the whole extra duration runs at that cluster's
highest recorded power, and each reading averages 7 s, so a brief peak above it is possible.


In [16]:
_fm_rows = []
for _fm_arm in _FM_ARMS:
    for _fm_col, _fm_lab in (("gc_explicit_per_s", "explicit collections per second"), ("gc_explicit_ms_per_s", "their duration, ms per second"),
                             ("gc_heap_mb_median", "heap after a collection, MB (median)")):
        _fm_rows.append(_fm_row(_fm_log, _fm_col, _fm_arm, technique=T.name(_fm_arm), measure=_fm_lab))
_fm_gc = pd.DataFrame(_fm_rows)
_fm_gcb, _fm_gca = _fm_log["gc_explicit_10s_before"].mean(), _fm_log["gc_explicit_10s_after"].mean()
_fm_ph = pd.read_csv(T.TABLES / "phase_durations_and_power.csv")
_fm_wl_w = float(_fm_ph[(_fm_ph["technique"] == "freeRASP") & (_fm_ph["measure"] == "active-phase mean power")]["delta_abs"].iloc[0])
_fm_extra_ms = float(_fm_gc[(_fm_gc["technique"] == "freeRASP") & (_fm_gc["measure"].str.startswith("their duration"))]["delta"].iloc[0])
_fm_pr69 = T.pairs(series[series["phone"].isin(["pixel6", "pixel9"])], "freerasp_debug", "workload_avg_watts")
_fm_wl_w69 = float(_fm_pr69["delta_abs"].mean())       # the ceiling comes from these two devices' rails, so compare like with like
_fm_wl_dev = {_p: float(T.pairs(series[series["phone"] == _p], "freerasp_debug", "workload_avg_watts")["delta_abs"].mean())
              for _p in ("pixel6", "pixel9")}
_fm_bound = []
for _fm_cl, _fm_cll in (("cpu_big", "big cluster"), ("cpu_mid", "middle cluster"), ("cpu_little", "little cluster")):
    for _fm_p in ("pixel6", "pixel9"):
        _fm_pmax = _fm_log.loc[_fm_log["phone"] == _fm_p, f"rail_max_{_fm_cl}"].max()
        _fm_bound.append(dict(technique="freeRASP", measure=f"bound on the extra collections' power, {_fm_cll}, {_FM_DEV[_fm_p]}",
                              cluster_highest_reading_w=_fm_pmax, bound_w=_fm_extra_ms / 1000 * _fm_pmax,
                              share_of_workload_difference_pct=100 * _fm_extra_ms / 1000 * _fm_pmax / _fm_wl_w,
                              share_of_p6_p9_workload_difference_pct=100 * _fm_extra_ms / 1000 * _fm_pmax / _fm_wl_w69,
                              device_workload_difference_w=_fm_wl_dev[_fm_p]))
_fm_gc = pd.concat([_fm_gc, pd.DataFrame(_fm_bound)], ignore_index=True)
_fm_gc["note"] = (f"explicit collections per cell: {_fm_gcb:.2f} in the 10 s before measurement, {_fm_gca:.2f} in its first 10 s; "
                  f"freeRASP's workload power difference {_fm_wl_w:.4f} W over all three devices and {_fm_wl_w69:.4f} W over the "
                  f"Pixel 6 and 9 ({_fm_wl_dev['pixel6']:+.4f} W and {_fm_wl_dev['pixel9']:+.4f} W)")
_fm_gc.to_csv(T.TABLES / "profiler_gc_load.csv", index=False)
T.register("tables/profiler_gc_load.csv", "REPO", "8 Threats (instrument load)",
           "collections forced by the memory sampler: rate, duration and heap per technique (pairs); an upper bound on the extra collections' power against freeRASP's workload difference", NB)
print(_fm_gc.round(3).to_string(index=False))


technique                                                        measure  n_pairs  control_mean  delta  ci_lo  ci_hi  pairs_higher_pct  cluster_highest_reading_w  bound_w  share_of_workload_difference_pct  share_of_p6_p9_workload_difference_pct  device_workload_difference_w                                                                                                                                                                                                                           note
 freeRASP                                explicit collections per second     78.0         0.753 -0.023 -0.030 -0.015            15.385                        NaN      NaN                               NaN                                     NaN                           NaN explicit collections per cell: 0.01 in the 10 s before measurement, 7.27 in its first 10 s; freeRASP's workload power difference 0.0964 W over all three devices and 0.0548 W over the Pixel 6 and 9 (+0.1107 W and -0.0010 W)
 fre

### The supplementary table and the per-second figure

`table6_freerasp_postlaunch` (SUPPLEMENT, Table S11) and `figures/freerasp_postlaunch_profile` (BODY).


In [17]:
def _fm_fmt(r, d=2):
    if pd.isna(r.get("delta")):
        return ""
    v = [round(r[k], d) + 0.0 for k in ("delta", "ci_lo", "ci_hi")]      # + 0.0: no "-0.00"
    return f"{v[0]:+.{d}f} [{v[1]:+.{d}f}, {v[2]:+.{d}f}]"


_fm_t = []
for _fm_s in ("0_4", "4_15", "15_end", "post_launch", "workload"):
    _fm_t.append({"": _FM_SEG[_fm_s].replace(" (the burst)", ", the burst"),
                  "All 78 pairs": _fm_fmt(_fm_row(_fm, f"e_{_fm_s}_j", "freerasp_debug")),
                  "The other eight": _fm_fmt(_fm_row(_fm[_fm["app"] != "medtimer"], f"e_{_fm_s}_j", "freerasp_debug")),
                  "medtimer": _fm_fmt(_fm_row(_fm[_fm["app"] == "medtimer"], f"e_{_fm_s}_j", "freerasp_debug"))})
_fm_r69 = _fm[_fm["phone"].isin(["pixel6", "pixel9"])]
for _fm_w, _fm_wl in (("post_launch", "passive phase"), ("burst_4_15", "4-15 s")):
    for _fm_m in ("battery", "cpu_total", "cpu_big", "gpu"):
        _fm_t.append({"": f"{_FM_RM[_fm_m]}, {_fm_wl}",
                      "All 78 pairs": _fm_fmt(_fm_row(_fm_r69, f"rail_{_fm_w}_{_fm_m}", "freerasp_debug")),
                      "The other eight": _fm_fmt(_fm_row(_fm_r69[_fm_r69["app"] != "medtimer"], f"rail_{_fm_w}_{_fm_m}", "freerasp_debug")),
                      "medtimer": _fm_fmt(_fm_row(_fm_r69[_fm_r69["app"] == "medtimer"], f"rail_{_fm_w}_{_fm_m}", "freerasp_debug"))})
_fm_t = pd.DataFrame(_fm_t)
_fm_np = {c: _fm_row(d, "e_post_launch_j", "freerasp_debug")["n_pairs"] for c, d in
          (("all", _fm), ("eight", _fm[_fm["app"] != "medtimer"]), ("medtimer", _fm[_fm["app"] == "medtimer"]))}
_fm_npr = {c: _fm_row(d, "rail_post_launch_battery", "freerasp_debug")["n_pairs"] for c, d in
           (("all", _fm_r69), ("eight", _fm_r69[_fm_r69["app"] != "medtimer"]), ("medtimer", _fm_r69[_fm_r69["app"] == "medtimer"]))}
assert _fm_np["all"] == 78, _fm_np
_fm_t.columns = ["", f"All pairs ({_fm_np['all']})", f"The other eight ({_fm_np['eight']})", f"medtimer ({_fm_np['medtimer']})"]
_fm_bt = pd.read_csv(T.TABLES / "freerasp_burst_timing.csv").set_index("device")
_fm_fcp = pd.read_csv(T.TABLES / "freerasp_fixed_cost_per_app.csv")
_fm_fcs = pd.read_csv(T.TABLES / "freerasp_fixed_cost_summary.csv")
_fm_med_there = float(_fm_fcp[(_fm_fcp["app"] == "medtimer") & (_fm_fcp["window"] == "startup")]["delta_j"].iloc[0])
_fm_8_there = float(_fm_fcs[(_fm_fcs["window"] == "startup") & _fm_fcs["subset"].str.startswith("eight")]["mean_delta_j"].iloc[0])
_fm_med_here = _fm_row(_fm[_fm["app"] == "medtimer"], "e_post_launch_j", "freerasp_debug")["delta"]
_fm_8_here = _fm_row(_fm[_fm["app"] != "medtimer"], "e_post_launch_j", "freerasp_debug")["delta"]
_fm_note10 = ("Not pre-specified: power rails, other processes' logs and traffic counters were analysed after the fact. The 4-15 s "
              "segment was chosen after inspecting the per-second profile averaged over the devices; on the Pixel 3 the burst "
              f"comes later, at {int(_fm_bt.loc['Pixel 3', 'core_from_s'])} to {int(_fm_bt.loc['Pixel 3', 'core_to_s'])} s, so the "
              "segment holds only its first part. Rails: the On-Device Power Monitor readings the thermal service logs every "
              "7 s; the Pixel 3 has no such log. Energies are integrated from each cell's battery samples by the mechanism "
              "extraction and averaged over pairs, so the window totals differ slightly from the per-application overheads, "
              "which use the run tables' window energies and average over applications: medtimer "
              f"{_fm_med_here:.2f} J here against {_fm_med_there:.2f} J, the other eight {_fm_8_here:.2f} J against {_fm_8_there:.2f} J.")
T.register(T.write_tex(
    _fm_t, "table6_freerasp_postlaunch",
    caption=("Where freeRASP's passive-phase energy goes (exploratory). Extra energy per cell (J) by segment of the "
             "passive phase and in the active phase, and extra power (W) on the Pixel~6 and~9 power rails; "
             "freeRASP minus control, mean of matched pairs with 95\\% paired $t$-interval. medtimer is the one "
             "freeRASP build without the INTERNET permission. Its passive phase contains its six-screen introduction, so "
             "medtimer is left out of the passive-phase analysis; the ``All pairs'' column, which includes it, is shown for "
             "completeness. Bold: the values the thesis states in words."),
    label="tab:freerasp-postlaunch", colfmt="lrrr", star=True,
    bold_cells={(int(_fm_t.index[_fm_t[""] == lab][0]), 2) for lab in ("4-15 s, the burst", "whole passive phase")}, blocks={0: "Extra energy per cell (J); seconds from the first energy sample",
                                                         5: f"Extra power (W), Pixel 6 and 9 rails (pairs: {_fm_npr['all']}, {_fm_npr['eight']}, {_fm_npr['medtimer']})"},
    notes=_fm_note10),
    "SUPPLEMENT", "Supplement (freeRASP's mechanism, exploratory)",
    "freeRASP's extra passive-phase energy by segment and extra power on the Pixel 6 and 9 rails; all pairs, the other eight, medtimer", NB)
_fm_tex = T.TABLES / "table6_freerasp_postlaunch.tex"            # en dashes for the ranges, in the LaTeX only
_fm_tex.write_text(_fm_tex.read_text().replace("0-4 s", "0--4 s").replace("4-15 s", "4--15 s"))
print(_fm_t.to_string(index=False))

# the per-second figure: freeRASP over the eight applications of the passive analysis with its interval, medtimer's
# freeRASP line drawn separately (no burst), and Bangcle; the 4-15 s band labelled; the legend below the axes
_fm_fig, _fm_ax = plt.subplots(figsize=(T.FIG_SINGLE, 2.15), layout="constrained")
_fm_ax.axvspan(4, 15, color="0.93", zorder=0)
_fm_ax.text(12.9, 0.98, "4–15 s", transform=_fm_ax.get_xaxis_transform(), ha="center", va="top", fontsize=7, color="0.35")
_fm_a = _fm_prof[(_fm_prof["technique"] == "freeRASP") & (_fm_prof["device"] == "all devices")]
_fm_ax.fill_between(_fm_a["second_from"] + 0.5, _fm_a["ci_lo"], _fm_a["ci_hi"], color=T.COLOURS["freeRASP"], alpha=0.18, lw=0, zorder=2)
_fm_ax.plot(_fm_a["second_from"] + 0.5, _fm_a["delta"], color=T.COLOURS["freeRASP"], lw=1.5, zorder=4,
            label=f"freeRASP, eight applications ({int(_fm_a['n_pairs'].max())} pairs, 95% CI)")
_fm_m = _fm_prof[(_fm_prof["technique"] == "freeRASP") & (_fm_prof["device"] == "all devices, medtimer only")]
_fm_ax.plot(_fm_m["second_from"] + 0.5, _fm_m["delta"], color=T.COLOURS["freeRASP"], lw=1.0, ls="--", marker="o", ms=2.4, markevery=3,
            mfc="white", zorder=3, label=f"freeRASP, medtimer ({int(_fm_m['n_pairs'].max())} pairs)")
_fm_b = _fm_prof[(_fm_prof["technique"] == "Bangcle") & (_fm_prof["device"] == "all devices")]
_fm_ax.plot(_fm_b["second_from"] + 0.5, _fm_b["delta"], color=T.COLOURS["Bangcle"], lw=1.0, zorder=3,
            label=f"Bangcle, eight applications ({int(_fm_b['n_pairs'].max())} pairs)")
T.control_line(_fm_ax, "y", outside=True, label="control")
_fm_ax.set_xlim(0, 30)
_fm_ax.set_xlabel("seconds after the first energy sample (passive phase)")
_fm_ax.set_ylabel("extra power vs.\ncontrol (W)")
_fm_ax.spines[["top", "right"]].set_visible(False)
_fm_fig.legend(loc="outside lower center", ncol=1, frameon=False, handlelength=2.4)
T.register(T.save_figure(_fm_fig, "freerasp_postlaunch_profile", T.FIG_SINGLE), "BODY", "Discussion (freeRASP's mechanism, exploratory)",
           "extra power in each second of the passive phase, matched pairs pooled over the devices: freeRASP over the eight applications of the "
           "passive analysis (95% interval), medtimer's freeRASP pairs separately, and Bangcle; the 4-15 s band labelled", NB)

T.record("mechanism.postlaunch_burst",
         burst_segment_j=[_fm_all.loc[_FM_SEG["4_15"], k] for k in ("delta", "ci_lo", "ci_hi")],
         whole_window_j=[_fm_all.loc[_FM_SEG["post_launch"], k] for k in ("delta", "ci_lo", "ci_hi")],
         burst_share_pct=_fm_burst_share, n_pairs=int(_fm_all.loc[_FM_SEG["4_15"], "n_pairs"]),
         cpu_rails_post_launch_w=[_fm_r.loc[("passive phase", "all CPU rails"), k] for k in ("delta", "ci_lo", "ci_hi")],
         battery_post_launch_w_pixel6_9=[_fm_r.loc[("passive phase", "battery"), k] for k in ("delta", "ci_lo", "ci_hi")],
         cpu_share_of_extra_power_pct={"post_launch": _fm_cpu_share_pl, "burst_4_15": _fm_cpu_share_b},
         basis="analysed series cells, the passive phase without medtimer; exploratory, not pre-specified")


                                      All pairs (78)    The other eight (69)           medtimer (9)
                       0-4 s    +1.65 [+1.35, +1.95]    +1.67 [+1.36, +1.99]   +1.44 [+0.28, +2.61]
           4-15 s, the burst +16.55 [+14.18, +18.92] +18.57 [+16.32, +20.83]   +1.01 [-0.43, +2.46]
15 s to the end of the phase    +2.60 [+1.43, +3.78]    +2.06 [+0.83, +3.28]  +6.80 [+3.49, +10.10]
         whole passive phase +20.80 [+18.79, +22.81] +22.31 [+20.37, +24.24]  +9.25 [+4.05, +14.46]
                active phase  +10.56 [+6.25, +14.87]  +10.60 [+5.73, +15.46] +10.29 [+5.68, +14.90]
      battery, passive phase    +0.81 [+0.68, +0.94]    +0.90 [+0.78, +1.03]   +0.06 [-0.03, +0.15]
all CPU rails, passive phase    +0.60 [+0.51, +0.69]    +0.67 [+0.58, +0.75]   +0.06 [+0.00, +0.12]
  big cluster, passive phase    +0.40 [+0.33, +0.46]    +0.44 [+0.37, +0.50]   +0.07 [+0.05, +0.09]
          GPU, passive phase    +0.00 [+0.00, +0.00]    +0.00 [+0.00, +0.00]   +0.00 [+0.00, +0.00]


## §6.4 (RQ3): Frame timing (exploratory)

Table 1 lists frame time and delayed frames among the RQ3 measures. Android Runner's frame-time profiler gives each cell a
median and a 95th-percentile frame time and a count of delayed frames (`frame_time_median_ms`, `frame_time_p95_ms`,
`delayed_frames_count`). Matched pairs as for energy: the difference in each measure, in its unit and in percent, with its
paired t-interval and the pairs in which the protected build is higher. Exploratory, no p.

In [18]:
_ft_cols = {"frame_time_median_ms": "median frame time (ms)", "frame_time_p95_ms": "95th-percentile frame time (ms)",
            "delayed_frames_count": "delayed frames per cell"}
_ft_ext = pd.concat(pd.read_csv(T.RUN_TABLES[_ft_b], usecols=["app", "phone", "variant", "rep", "cell_start_ts_utc", *_ft_cols],
                                dtype=str).assign(batch=_ft_b) for _ft_b in T.SERIES)
_ft = series.merge(_ft_ext, on=["app", "phone", "variant", "rep", "batch", "cell_start_ts_utc"], how="left", validate="one_to_one")
for _ft_c in _ft_cols:
    _ft[_ft_c] = pd.to_numeric(_ft[_ft_c], errors="coerce")
_ft_rows = []
for _ft_arm in T.ARMS:
    for _ft_c, _ft_label in _ft_cols.items():
        _ft_p = T.pairs(_ft.dropna(subset=[_ft_c]), _ft_arm, _ft_c)
        _ft_m, _ft_lo, _ft_hi, _ = T.ci_naive(_ft_p["delta_abs"])
        _ft_pp = _ft_p[np.isfinite(_ft_p["delta_pct"])]
        _ft_mp, _ft_lop, _ft_hip, _ = T.ci_naive(_ft_pp["delta_pct"])
        _ft_rows.append({"technique": T.name(_ft_arm), "measure": _ft_label, "pairs": len(_ft_p),
                         "control_mean": _ft_p["baseline"].mean(), "treated_mean": _ft_p["treated"].mean(),
                         "difference": _ft_m, "ci_lo": _ft_lo, "ci_hi": _ft_hi,
                         "difference_pct": _ft_mp, "pct_ci_lo": _ft_lop, "pct_ci_hi": _ft_hip,
                         "pairs_higher": int((_ft_p["delta_abs"] > 0).sum())})
_ft_tab = pd.DataFrame(_ft_rows)
_ft_tab.to_csv(T.TABLES / "frame_timing.csv", index=False)
T.register("tables/frame_timing.csv", "REPO", "6.4 RQ3 (frame timing; exploratory)",
           "matched-pair change in median and 95th-percentile frame time and in delayed frames, per technique", NB)
with pd.option_context("display.width", 200):
    print(_ft_tab.round(2).to_string(index=False))

technique                         measure  pairs  control_mean  treated_mean  difference  ci_lo  ci_hi  difference_pct  pct_ci_lo  pct_ci_hi  pairs_higher
  Bangcle          median frame time (ms)     69          7.26          7.34        0.07  -0.02   0.16            0.78      -0.02       1.58            41
  Bangcle 95th-percentile frame time (ms)     69         27.47         27.48        0.00  -0.66   0.67           -0.58      -2.13       0.97            38
  Bangcle         delayed frames per cell     69        204.72        208.11        3.39  -0.82   7.60            0.53      -1.44       2.50            36
 Allatori          median frame time (ms)     45          7.41          7.37       -0.04  -0.15   0.07           -0.07      -0.93       0.80            27
 Allatori 95th-percentile frame time (ms)     45         27.23         27.46        0.23  -0.30   0.75            0.52      -1.16       2.20            24
 Allatori         delayed frames per cell     45        256.74        

## Discussion: does freeRASP's thread overhead depend on the application?

Per application: freeRASP's added logging threads beside the application's own size (`app_characteristics.csv`), its control
thread count, and the median number of lines freeRASP's own code (Talsec) logs per cell. Summarised for all nine applications and
without poetskingdom, with Spearman rank correlations against each size measure. Exploratory: direction and magnitude only, no
p-values (nine applications, several candidate measures).

poetskingdom's workload restarts the application inside every cell (`appium_android_tests/poetskingdom/scenarios.py`,
`_create_poem`: terminate and relaunch before the first poem; more restarts on recovery), and freeRASP starts with the application.
Two columns check that it started more often there: the application's process starts per cell, counted in the system log
(`launch_timing_series.csv`), and the lines freeRASP's own code logs per cell.

In [19]:
from scipy import stats as _st
ac = pd.read_csv(T.DATA / "inputs" / "app_characteristics.csv").set_index("app")
size_cols = ["app_classes", "app_methods", "dex_bytes", "apk_bytes"]
fr = th[th["technique"] == "freeRASP"].set_index("app")[["delta_threads", "control_threads"]]
fr = fr.join(ac[size_cols]).join(lc[lc["variant"] == "freerasp_debug"].groupby("app")["talsec_calls"].median()
                                 .rename("freerasp_log_lines_median"))
starts = pd.read_csv(T.DATA / "inputs" / "launch_timing_series.csv", dtype={"rep": str}).merge(series[key], on=key)
fr = fr.join(starts[starts["variant"] == "freerasp_debug"].groupby("app")["app_process_starts"].median()
             .rename("process_starts_per_cell_median"))
fr.reset_index().to_csv(T.TABLES / "freerasp_threads_vs_app.csv", index=False)
T.register("tables/freerasp_threads_vs_app.csv", "REPO", "Discussion (RQ3)",
           "freeRASP's added logging threads per application, beside app size, control threads and freeRASP's own log lines", NB)
rows = []
for label, d in (("all nine", fr), ("without poetskingdom", fr.drop(index="poetskingdom"))):
    x = d["delta_threads"]
    row = {"subset": label, "n_apps": len(d), "mean": x.mean(), "sd": x.std(), "cv_pct": 100 * x.std() / x.mean(),
           "min": x.min(), "max": x.max()}
    for col in size_cols + ["control_threads"]:
        row[f"rho_{col}"] = _st.spearmanr(x, d[col]).statistic
    rows.append(row)
tsum = pd.DataFrame(rows)
tsum.to_csv(T.TABLES / "freerasp_threads_summary.csv", index=False)
T.register("tables/freerasp_threads_summary.csv", "REPO", "Discussion (RQ3)",
           "freeRASP added logging threads: spread and Spearman rho with app size, all nine and without poetskingdom", NB)
print(fr.round(2).to_string()); print(); print(tsum.round(2).to_string(index=False))
w = tsum.set_index("subset").loc["without poetskingdom"]
others = fr.drop(index="poetskingdom")["freerasp_log_lines_median"]
T.record("discussion.freerasp_threads", range_without_poetskingdom=[w["min"], w["max"]], mean_without_poetskingdom=w["mean"],
         cv_pct_without_poetskingdom=w["cv_pct"], poetskingdom=fr.loc["poetskingdom", "delta_threads"],
         freerasp_log_lines_median_poetskingdom=fr.loc["poetskingdom", "freerasp_log_lines_median"],
         freerasp_log_lines_median_others=[others.min(), others.max()],
         process_starts_per_cell_poetskingdom=fr.loc["poetskingdom", "process_starts_per_cell_median"],
         process_starts_per_cell_others=[fr.drop(index="poetskingdom")["process_starts_per_cell_median"].min(),
                                         fr.drop(index="poetskingdom")["process_starts_per_cell_median"].max()])

              delta_threads  control_threads  app_classes  app_methods  dex_bytes  apk_bytes  freerasp_log_lines_median  process_starts_per_cell_median
app                                                                                                                                                    
calculator             4.47            15.71          108          591   11683256    9930850                        2.0                             1.0
documenter             7.90            21.30          189          816   14124084   57311416                        4.0                             1.0
itsok                  5.50             8.80          395         1754   12700372    6701427                        2.0                             1.0
keep_recipe            6.15            12.67          354         1418   16800544    8183993                        2.0                             1.0
linkhub                5.53            12.70          852         3477   36945180   1625

## Discussion: does the device gradient follow anything measured about the phones?

Per device, freeRASP against its control (matched pairs, series): the energy effect in both windows, in percent and in joules;
the control's own device-wide CPU load during the workload and what freeRASP adds to it (`ar_cpu_workload` in the mechanism input, the active phase only; Android Runner samples
the `TOTAL` line of `dumpsys cpuinfo`, all processes); the added logging threads; and the Android version each phone ran.
Three devices differ in system-on-chip, Android version and battery hardware at once, so no single property can be isolated;
the question is only whether any measured quantity moves with the gradient. Exploratory.

In [20]:
ext = pd.concat(pd.read_csv(T.RUN_TABLES[b], usecols=["app", "phone", "variant", "rep", "cell_start_ts_utc", "cpu_avg_pct",
                                                    "android_release"], dtype=str).assign(batch=b) for b in T.SERIES)
cells = series.merge(ext, on=["app", "phone", "variant", "rep", "batch", "cell_start_ts_utc"], how="left", validate="one_to_one")
cells["cpu_avg_pct"] = pd.to_numeric(cells["cpu_avg_pct"], errors="coerce")
cells = cells.merge(_fm[key + ["ar_cpu_workload"]], on=key, how="left", validate="one_to_one")   # active-phase CPU
thr_dev = (g["freerasp_debug"] - g["baseline_debug"]).dropna().groupby(level="phone").mean()
ppf = pd.read_csv(T.TABLES / "per_phone_freerasp.csv").set_index("phone")
rows = []
for ph in ("pixel3", "pixel6", "pixel9"):
    cc = cells[cells["phone"] == ph]
    row = {"device": T.PHONE_DISPLAY[ph], "android": cc["android_release"].mode().iloc[0]}
    for w, col in T.WINDOWS.items():
        pr = T.pairs(T.phase_set(cc, w), "freerasp_debug", col)   # the passive phase leaves out medtimer
        m, lo, hi, _ = T.ci_naive(pr["delta_pct"])
        row.update({f"{w}_pairs": len(pr), f"{w}_delta_pct": m, f"{w}_ci_lo": lo, f"{w}_ci_hi": hi,
                    f"{w}_control_j": pr["baseline"].mean(), f"{w}_delta_j": pr["delta_abs"].mean()})
    assert abs(row["workload_delta_pct"] - ppf.loc[T.PHONE_DISPLAY[ph], "delta_pct"]) < 1e-9, "differs from per_phone_freerasp.csv"
    pr = T.pairs(cc, "freerasp_debug", "ar_cpu_workload")      # the active phase only; cpu_avg_pct averages the whole run
    m, lo, hi, _ = T.ci_naive(pr["delta_abs"])
    row.update({"control_cpu_pct": pr["baseline"].mean(), "added_cpu_pp": m, "added_cpu_ci_lo": lo, "added_cpu_ci_hi": hi,
                "added_logging_threads": thr_dev[ph]})
    rows.append(row)
dv = pd.DataFrame(rows)
dv.to_csv(T.TABLES / "freerasp_by_device.csv", index=False)
T.register("tables/freerasp_by_device.csv", "REPO", "Discussion (device gradient)",
           "freeRASP per device: energy effect both windows (% and J; the passive phase without medtimer), control CPU load and added CPU load "
           "(active phase), added logging threads, Android version", NB)
print(dv.round(2).T.to_string())
for r in dv.itertuples():
    T.record(f"discussion.freerasp_device.{r.device.replace(' ', '').lower()}", android=r.android,
             workload_delta_pct=r.workload_delta_pct, workload_delta_j=r.workload_delta_j,
             startup_delta_pct=r.startup_delta_pct, startup_ci95=[r.startup_ci_lo, r.startup_ci_hi], startup_delta_j=r.startup_delta_j,
             control_cpu_pct=r.control_cpu_pct, added_cpu_pp=r.added_cpu_pp, added_cpu_ci95=[r.added_cpu_ci_lo, r.added_cpu_ci_hi],
             added_logging_threads=r.added_logging_threads)

                             0        1        2
device                 Pixel 3  Pixel 6  Pixel 9
android                     12       16       16
workload_pairs              24       27       27
workload_delta_pct        8.04     4.04     0.54
workload_ci_lo            5.41     2.48    -1.93
workload_ci_hi           10.67     5.61     3.01
workload_control_j      265.12   330.36   355.11
workload_delta_j         19.27    12.77     0.17
startup_pairs               21       24       24
startup_delta_pct        32.37    43.97    19.11
startup_ci_lo            27.23    38.25    14.89
startup_ci_hi             37.5    49.69    23.33
startup_control_j        64.06    75.31    91.37
startup_delta_j           19.0    30.58    16.92
control_cpu_pct          11.69    13.78    18.86
added_cpu_pp              2.23     1.96     2.43
added_cpu_ci_lo           1.43     0.78     0.99
added_cpu_ci_hi           3.02     3.15     3.87
added_logging_threads     4.27     7.68     8.12


## §6.4 (RQ3): Channels in which each technique shows

Is a technique visible at all, outside energy? Per technique, against its control on the same cells: the application's memory
while the profilers run (`mem_pss_avg_mb`, Android Runner's `dumpsys meminfo` TOTAL PSS for the package; matched pairs), and
two log channels from `logcat_activity_*.csv`: lines from Bangcle's runtime stub (`com.SecShell.*`) and hidden-API access
lines, per cell. The cold-launch time is in `launch_time.csv` (Phase_Analysis).

In [21]:
mem = pd.concat(pd.read_csv(T.RUN_TABLES[b], usecols=["app", "phone", "variant", "rep", "cell_start_ts_utc", "mem_pss_avg_mb"],
                            dtype=str).assign(batch=b) for b in T.SERIES)
cm = series.merge(mem, on=["app", "phone", "variant", "rep", "batch", "cell_start_ts_utc"], how="left", validate="one_to_one")
cm["mem_pss_avg_mb"] = pd.to_numeric(cm["mem_pss_avg_mb"], errors="coerce")
ctrl_logs = lc[lc["variant"] == "baseline_debug"]
rows = []
for arm in T.ARMS:
    pr = T.pairs(cm, arm, "mem_pss_avg_mb")
    m, lo, hi, _ = T.ci_naive(pr["delta_pct"])
    tl = lc[lc["variant"] == arm]
    rows.append({"technique": T.name(arm), "memory_pairs": len(pr), "memory_delta_pct": m, "memory_ci_lo": lo, "memory_ci_hi": hi,
                 "memory_delta_mb": pr["delta_abs"].mean(), "memory_pairs_higher": int((pr["delta_pct"] > 0).sum()),
                 "secshell_lines_per_cell": tl["secshell_calls"].mean(), "cells_with_secshell_pct": 100 * (tl["secshell_calls"] > 0).mean(),
                 "hiddenapi_lines_per_cell": tl["hiddenapi_calls"].mean(),
                 "control_hiddenapi_lines_per_cell": ctrl_logs["hiddenapi_calls"].mean()})
    for _ph in ("pixel3", "pixel6", "pixel9"):          # the Pixel 3 (Android 12) logs no hidden-API notice for any build
        rows[-1][f"cells_with_secshell_pct_{_ph}"] = 100 * (tl.loc[tl["phone"] == _ph, "secshell_calls"] > 0).mean()
        rows[-1][f"cells_with_hiddenapi_pct_{_ph}"] = 100 * (tl.loc[tl["phone"] == _ph, "hiddenapi_calls"] > 0).mean()
        rows[-1][f"control_cells_with_hiddenapi_pct_{_ph}"] = 100 * (ctrl_logs.loc[ctrl_logs["phone"] == _ph, "hiddenapi_calls"] > 0).mean()
ch = pd.DataFrame(rows)
ch.to_csv(T.TABLES / "runtime_channels.csv", index=False)
T.register("tables/runtime_channels.csv", "REPO", "6.4 RQ3 (channels)",
           "per technique: memory vs control (matched pairs), Bangcle-stub log lines and hidden-API lines per cell", NB)
print(ch.round(2).T.to_string())
for r in ch.itertuples():
    T.record(f"channels.{r.technique.lower()}", memory_delta_pct=r.memory_delta_pct, memory_ci95=[r.memory_ci_lo, r.memory_ci_hi],
             memory_delta_mb=r.memory_delta_mb, memory_pairs_higher=f"{r.memory_pairs_higher} of {r.memory_pairs}",
             secshell_lines_per_cell=r.secshell_lines_per_cell, cells_with_secshell_pct=r.cells_with_secshell_pct,
             hiddenapi_lines_per_cell=r.hiddenapi_lines_per_cell, control_hiddenapi_lines_per_cell=r.control_hiddenapi_lines_per_cell)

                                               0         1         2
technique                                Bangcle  Allatori  freeRASP
memory_pairs                                  69        45        78
memory_delta_pct                            12.0     -0.05      14.6
memory_ci_lo                                9.25      -0.3     13.82
memory_ci_hi                               14.74       0.2     15.39
memory_delta_mb                            16.96     -0.07      21.5
memory_pairs_higher                           69        27        78
secshell_lines_per_cell                     3.54       0.0       0.0
cells_with_secshell_pct                    70.56       0.0       0.0
hiddenapi_lines_per_cell                    6.86       2.1       5.8
control_hiddenapi_lines_per_cell            3.34      3.34      3.34
cells_with_secshell_pct_pixel3               0.0       0.0       0.0
cells_with_hiddenapi_pct_pixel3              0.0       0.0       0.0
control_cells_with_hiddenapi_pct_p

In [22]:
# What each protection adds to its control's manifest. The parity table tracks INTERNET only; the integration adds more.
_pm = pd.read_csv(T.DATA / "inputs" / "variant_permissions.csv")
_pm = _pm[_pm["app"].isin(series["app"].unique())]          # the analysed applications; the parity table has only these
_pm_par = pd.read_csv(T.TABLES / "variant_parity_full.csv")
_pm_sha = _pm.drop_duplicates(["app", "variant"]).merge(_pm_par[["app", "variant", "apk_sha256"]].drop_duplicates(),
                                                       on=["app", "variant"], suffixes=("", "_parity"))
assert len(_pm_sha) == _pm[["app", "variant"]].drop_duplicates().shape[0], "a build is missing from the parity table"
assert (_pm_sha["apk_sha256"] == _pm_sha["apk_sha256_parity"]).all(), "permissions read from another APK than the parity table's"
_pm_apps = sorted(series["app"].unique())
_pm_set = lambda a, v: set(_pm[(_pm["app"] == a) & (_pm["variant"] == v)]["permission"])
_pm_rows = [dict(technique=T.name(_pm_arm), app=_pm_a, permission=_pm_p)
            for _pm_arm in T.ARMS for _pm_a in _pm_apps if _pm_set(_pm_a, _pm_arm)
            for _pm_p in sorted(_pm_set(_pm_a, _pm_arm) - _pm_set(_pm_a, "baseline_debug"))]
_pm_added = pd.DataFrame(_pm_rows, columns=["technique", "app", "permission"])
_pm_sum = (_pm_added.groupby(["technique", "permission"])["app"]
           .agg(n_apps="count", apps=lambda s: ", ".join(sorted(s))).reset_index()
           .sort_values(["technique", "n_apps", "permission"], ascending=[True, False, True]))
_pm_sum["of_apps"] = len(_pm_apps)
_pm_sum.to_csv(T.TABLES / "added_permissions.csv", index=False)
T.register("tables/added_permissions.csv", "REPO", "4.2 / 7.3 (what each protection adds to the manifest)",
           "permissions each protected build declares that its control does not, with the applications (analysed applications)", NB)
print(_pm_sum.to_string(index=False))

technique                                                 permission  n_apps                                                                                            apps  of_apps
  Bangcle                       android.permission.ACCESS_WIFI_STATE       9 calculator, documenter, itsok, keep_recipe, linkhub, medtimer, pdfviewer, poetskingdom, tipuous        9
  Bangcle                    android.permission.ACCESS_NETWORK_STATE       6                                 calculator, itsok, keep_recipe, linkhub, medtimer, poetskingdom        9
 freeRASP                       android.permission.ACCESS_WIFI_STATE       9 calculator, documenter, itsok, keep_recipe, linkhub, medtimer, pdfviewer, poetskingdom, tipuous        9
 freeRASP com.google.android.providers.gsf.permission.READ_GSERVICES       9 calculator, documenter, itsok, keep_recipe, linkhub, medtimer, pdfviewer, poetskingdom, tipuous        9
 freeRASP                           android.permission.USE_BIOMETRIC       8           cal

## §6.4 (RQ3): Static code effects of obfuscation

What Allatori did to each application's own code, with its injected string decryptors separated out, for the
applications in the Allatori arm (itsok's build is excluded by the build-type rule: a release build's code differs from a
debug build's for reasons other than obfuscation). Renaming extent is compared against application size.

In [23]:
st_ = pd.read_csv(T.DATA / "inputs" / "allatori_static_code_effects.csv")
arm_apps = sorted(df.loc[df["variant"] == "allatori_debug", "app"].unique())
st_ = st_[st_["app"].isin(arm_apps)]
size = pd.read_csv(T.DATA / "inputs" / "app_characteristics.csv")[["app", "app_classes"]]
st_ = st_.merge(size, on="app")
st_.to_csv(T.TABLES / "allatori_static_code_effects.csv", index=False)
T.register("tables/allatori_static_code_effects.csv", "REPO", "6.4 RQ3 (mechanism)", "Allatori: classes renamed, decryptors, call sites, change in the app's own code", NB)
print(st_.to_string(index=False))
from scipy import stats as _st
r, p = _st.pearsonr(st_["app_classes"], st_["classes_renamed"])
print(f"\nr(app classes, classes renamed) = {r:+.3f} over {len(st_)} applications (p = {p:.3f})")
T.record("mechanism.allatori_renaming_vs_size", r=r, p=p, n_apps=len(st_))

         app  classes_renamed  decryptor_routines  decrypt_call_sites  app_code_instructions_pct  app_code_branches_pct  app_code_goto_pct  app_code_if_pct  app_classes
  calculator               61                  14                1033                      16.45                  -4.43             -13.57             0.12          108
    medtimer              400                  32                1615                       9.48                  -8.13             -25.00             0.30          729
   pdfviewer               25                   2                 112                       7.45                  -1.00              -2.70             0.00           51
poetskingdom              490                   9                3082                      -1.49                  -3.30              -8.58            -0.06          608
     tipuous              138                  10                 697                       1.63                  -3.60              -9.22            -0.42

## Figure 3: mechanism panel

In [24]:
aud = pd.read_csv(T.TABLES / "auditd_discriminator.csv").set_index("technique")
th = pd.read_csv(T.TABLES / "thread_deltas_per_app.csv")
techs = ["Bangcle", "Allatori", "freeRASP"]
y = np.arange(len(techs))[::-1]                     # Bangcle at the top, the order of every other figure
fig, (a1, a2) = plt.subplots(1, 2, figsize=(T.FIG_SINGLE, 2.0), layout="constrained", sharey=True,
                             gridspec_kw={"width_ratios": [1, 1.15]})
# (a) a rate, not a difference: each technique's share of cells with denials. The control is not a bar;
# it is the labelled reference line at its own rate (0 of its cells).
ctrl = aud.loc["Control"]
for yi, t in zip(y, techs):
    r = aud.loc[t]
    a1.barh(yi, r.share_pct, color=T.COLOURS[t], height=0.55, zorder=3)
    txt = f"{int(r.with_auditd)} of {int(r.cells)}"
    if r.share_pct > 50:
        a1.text(r.share_pct - 3, yi, txt, ha="right", va="center", color="white", zorder=4)
    else:
        a1.text(max(r.share_pct, ctrl.share_pct) + 4, yi, txt, ha="left", va="center", zorder=4)   # right of the control line
T.control_line(a1, "x", outside=True, value=ctrl.share_pct,
               label=f"unprotected control:\n{int(ctrl.with_auditd)} of {int(ctrl.cells)} cells")
a1.set_xlim(-8, 104)
a1.set_xlabel("(a) Cells with SELinux\ndenials (%)")
# (b) a difference: per application, threads vs. the same application's control on the same device
for yi, t in zip(y, techs):
    d = th[th["technique"] == t]
    a2.scatter(d["delta_threads"], np.full(len(d), yi) + np.linspace(-0.18, 0.18, len(d)), s=7,
               color=T.COLOURS[t], zorder=3)
T.control_line(a2, "x", outside=True, label="unprotected\ncontrol")
a2.set_xlim(-2, 15)
a2.set_xlabel("(b) Δ threads vs. control\n(per application)")
a1.set_yticks(y); a1.set_yticklabels(techs)
a1.set_ylim(-0.6, len(techs) - 0.4)
for a in (a1, a2):
    a.spines[["top", "right", "left"]].set_visible(False)
    a.tick_params(axis="y", length=0)
T.register(T.save_figure(fig, "figure3_mechanism", T.FIG_SINGLE), "REPO", "6.4 RQ3 (mechanism)",
           "(a) share of analysed cells with SELinux denials per technique, control as labelled reference line (b) threads vs. control per application", NB)